In [ ]:
# NLG API-first: OpenAI dari .env, Ollama Docker sebagai fallback otomatis.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "modules").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.hostage_nlg import build_nlg_provider

NLG_PROVIDER = os.getenv("NLG_PROVIDER", "api")
NLG_API_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
NLG_OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:8b")
NLG_OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11435")
llm, nlg_provider_config = build_nlg_provider(
    provider=NLG_PROVIDER,
    api_model=NLG_API_MODEL,
    ollama_model=NLG_OLLAMA_MODEL,
    ollama_base_url=NLG_OLLAMA_URL,
)
print(f"NLG HOSTAGE siap | utama={nlg_provider_config['primary_provider']} | fallback={nlg_provider_config['fallback_provider']}")

In [ ]:
# Fuzzy bersama: seluruh notebook memakai aturan yang sama dari modules/.
from pathlib import Path
import joblib
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "modules").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.hostage_fuzzy import calculate_bluff_indicator, describe_bluff_level

def _build_hostage_prompt(chat_pemain, data_npc, intent, bluff_score):
    bluff_level = describe_bluff_level(bluff_score)
    if bluff_score >= 70:
        mood_instruction = "Kamu sangat curiga. Minta alibi atau soroti kontradiksi secara tegas."
    elif bluff_score >= 40:
        mood_instruction = "Kamu waspada. Tanyakan detail klaim dan jangan langsung percaya."
    else:
        mood_instruction = "Kamu tenang. Tanggapi dengan netral, tetapi tetap perhatikan alur diskusi."

    return (
        f"Kamu adalah NPC dalam game social deduction HOSTAGE.\n"
        f"Fase: {data_npc['phase']}, putaran: {data_npc['round']}.\n"
        f"Pemain berkata: {chat_pemain}\n"
        f"Intent pemain: {intent}. Indikasi bluff dari data publik: {bluff_level} ({bluff_score:.1f}/100).\n"
        f"{mood_instruction}\n"
        "Balas hanya satu kalimat singkat dalam bahasa Indonesia gamer. Jangan menyebut data sistem atau role rahasia."
    )

def npc_respond(chat_pemain, data_npc):
    intent = get_user_intent(chat_pemain)
    bluff_score = calculate_bluff_indicator(
        data_npc['accusation_count'],
        data_npc['claim_contradiction'],
        data_npc['public_evidence'],
    )
    prompt = _build_hostage_prompt(chat_pemain, data_npc, intent, bluff_score)
    response = llm.invoke(prompt)

    return {
        'intent_detected': intent,
        'bluff_indicator': round(bluff_score, 2),
        'bluff_level': describe_bluff_level(bluff_score),
        'npc_reply': response.content.strip(),
    }

# Integrasi final NLU -> Fuzzy -> NLG memakai modul bersama.
from modules.hostage_nlg import GameMemory, generate_npc_response

GAME_MEMORY = GameMemory()
INTENT_MODEL = None

def _resolve_intent_model_path():
    model_filenames = (
        "intent_classifier_svm_tuned.pkl",
        "intent_classifier_svm.pkl",
        "intent_classifier_nb_tuned.pkl",
        "intent_classifier_nb.pkl",
        "intent_classifier.pkl",  # kompatibilitas model lama
    )
    candidates = [base / "models" / filename for base in (Path.cwd(), PROJECT_ROOT) for filename in model_filenames]
    return next((path for path in candidates if path.exists()), None)

def get_user_intent(user_text):
    global INTENT_MODEL
    if INTENT_MODEL is None:
        model_path = _resolve_intent_model_path()
        if model_path is None:
            return "neutral"
        INTENT_MODEL = joblib.load(model_path)
    return str(INTENT_MODEL.predict([user_text])[0])

def reset_game_memory():
    GAME_MEMORY.entries.clear()

def npc_respond(chat_pemain, game_state, npc_name="NPC", speaker_pemain="Pemain"):
    intent = get_user_intent(chat_pemain)
    return generate_npc_response(
        llm=llm,
        chat_pemain=chat_pemain,
        intent=intent,
        game_state=game_state,
        npc_name=npc_name,
        memory=GAME_MEMORY,
        speaker_pemain=speaker_pemain,
    )

In [27]:
if __name__ == "__main__":
    reset_game_memory()
    GAME_MEMORY.add("Dimas", "Naya tadi minta alibi B sebelum Gag Order dipakai.", phase="diskusi", round_number=3, target="Naya")
    game_state = {
        "phase": "diskusi",
        "round": 3,
        "accusation_count": 7,
        "claim_contradiction": 8,
        "public_evidence": 7,
        "silence_anomaly": 4,
        "vote_pressure": 6,
        "public_events": ["Gag Order dipakai saat diskusi.", "Tidak ada target Hostage yang diumumkan sistem."],
        "silent_players": ["B"],
    }
 
    input_chat = "Si B kena Gag Order pas mau jelasin alibi; menurut gw dia patut dicurigai sebagai Hitman."
 
    hasil = npc_respond(input_chat, game_state, npc_name="Naya", speaker_pemain="Raka")
 
    print(f"Chat Pemain  : {input_chat}")
    print(f"{'─'*40}")
    print(f"Intent       : {hasil['intent_detected']}")
    print(f"Indikasi Bluff : {hasil['bluff_indicator']:.2f}% ({hasil['bluff_level'].upper()})")
    print(f"{'─'*40}")
    print(f"NPC          : {hasil['npc_reply']}")
 

Chat Pemain  : Woy, si budi mencurigakan banget, koinnya tiba-tiba banyak!
────────────────────────────────────────
Intent       : deflecting
Threat Level : 50.00%
────────────────────────────────────────
NPC          : anjir lo curigain doang, koin gw nggak usah fix pinter-pinter lu
